# Recovery, WASM, And React Runtime Workflows

Examples for compacts, pruning, seed bootstrap, cold backups, WASM-safe helpers, React stores, browser live repositories, and unsupported native Git behavior in WASM.

Run this cell from the repository root after `npm ci` and `npm run build`. The stored output below was regenerated by `npm run notebooks:build`.

## Compact, Prune, Seed Bootstrap, And Cold Restore

**Use when:** Use this flow when a repository needs disaster recovery artifacts and a verified bootstrap path for fresh peers.

The next cell is the executable example.

In [1]:
import { mkdtempSync, writeFileSync, readdirSync } from 'node:fs';
import { tmpdir } from 'node:os';
import { join } from 'node:path';
import {
  EpochRepository,
  createCompact,
  pruneEventLogBeforeCompact,
  restoreFromCompact,
  createColdBackup,
  restoreFromColdBackup,
  verifyColdBackup,
  bootstrapFromSeed,
} from 'epoch';

const root = mkdtempSync(join(tmpdir(), 'epoch-notebook-ha-'));
const repository = EpochRepository.create(root, { author: 'seed' });
writeFileSync(join(root, 'one.txt'), 'one\n');
repository.recordFile('one.txt', 'text/plain');
const compact = createCompact(repository);
writeFileSync(join(root, 'two.txt'), 'two\n');
repository.recordFile('two.txt', 'text/plain');
const backup = createColdBackup(repository, { compact });
verifyColdBackup(backup);
pruneEventLogBeforeCompact(repository, compact.id);
const eventFilesAfterPrune = readdirSync(repository.eventsDir).length;
restoreFromCompact(repository, compact.id);
const restoredFromCompactEvents = repository.events().length;

const peerRoot = mkdtempSync(join(tmpdir(), 'epoch-notebook-seed-peer-'));
const peer = EpochRepository.create(peerRoot, { author: 'peer' });
const seedSync = await bootstrapFromSeed(peer, { peerId: 'seed', multiaddr: root, trustLevel: 'full' });

const recoveredRoot = mkdtempSync(join(tmpdir(), 'epoch-notebook-restore-'));
const recovered = EpochRepository.create(recoveredRoot, { author: 'recovered' });
restoreFromColdBackup(recovered, backup);

console.log(JSON.stringify({
  compactIdLength: compact.id.length,
  eventFilesAfterPrune,
  restoredFromCompactEvents,
  backupTailEvents: backup.tailEvents.length,
  seedSyncEvents: seedSync.eventsCopied,
  peerVerifyProblems: peer.verify().length,
  recoveredEvents: recovered.events().length,
  recoveredVerifyProblems: recovered.verify().length,
}, null, 2));

{
  "compactIdLength": 64,
  "eventFilesAfterPrune": 1,
  "restoredFromCompactEvents": 2,
  "backupTailEvents": 1,
  "seedSyncEvents": 1,
  "peerVerifyProblems": 0,
  "recoveredEvents": 2,
  "recoveredVerifyProblems": 0
}


**How to read the output:** The compact ID is content-addressed, pruning keeps only tail events, seed bootstrap copies verified state, and cold backup restore recovers compact plus tail history.

## WASM-Safe Helpers, React History, And Browser Live Repositories

**Use when:** Use this flow when browser code needs local state history, VFS-backed sync, and explicit rejection of native Git operations.

The next cell is the executable example.

In [2]:
import { CRDTRegistry, EntityType } from 'epoch';
import { EpochWasmGit } from 'epoch/Epoch.WASM.Git';
import {
  createEpochReactStore,
  createMemoryEpochReactStorage,
  createEpochLiveRepository,
  createMemoryEpochVfs,
} from 'epoch/Epoch.WASM.React';

const storage = createMemoryEpochReactStorage();
const store = createEpochReactStore({
  entity: 'counter',
  initialState: { count: 0 },
  storageKey: 'epoch:notebook:counter',
  storage,
});
store.setState((state) => ({ count: state.count + 1 }));
store.setState((state) => ({ count: state.count + 4 }));
const latest = store.getSnapshot();
store.rewind(1);
const rewound = store.getSnapshot();
const rematerialized = store.materialize('latest');

const vfsA = createMemoryEpochVfs();
const vfsB = createMemoryEpochVfs();
const liveA = createEpochLiveRepository({ vfs: vfsA, author: 'browser-a' });
const liveB = createEpochLiveRepository({ vfs: vfsB, author: 'browser-b' });
liveA.append('canvas', { widget: 'chart', x: 16 });
liveB.append('canvas', { widget: 'note', x: 40 });
const copied = liveA.syncFrom(vfsB);
const mergedText = CRDTRegistry.defaults().merge(EntityType.plainText, 'A\n', 'A\nB\n', 'A\nC\n');

let wasmGitError = '';
try {
  EpochWasmGit.execute('status');
} catch (error) {
  wasmGitError = error instanceof Error ? error.message : String(error);
}

console.log(JSON.stringify({
  latestCount: latest.state.count,
  rewoundCount: rewound.state.count,
  rematerializedCount: rematerialized.count,
  liveSyncEventsCopied: copied,
  liveEntities: liveA.view().entities,
  wasmMergedText: mergedText.trim().split('\n'),
  wasmGitError,
}, null, 2));

{
  "latestCount": 5,
  "rewoundCount": 0,
  "rematerializedCount": 5,
  "liveSyncEventsCopied": 1,
  "liveEntities": {
    "canvas": {
      "widget": "note",
      "x": 40
    }
  },
  "wasmMergedText": [
    "A",
    "B",
    "C"
  ],
  "wasmGitError": "git status is not supported by Epoch WASM Git compatibility: native Git repository access is unavailable in the WASM runtime; use the Core or CLI Git package on a host filesystem"
}


**How to read the output:** The React store can rewind and rematerialize history, browser peers exchange VFS events, WASM can reuse pure merge helpers, and native Git access fails with a clear runtime-specific error.